# AI Alvásasszisztens RAG kereső létrehozása és tesztelése


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Rag fájl beimportálása

In [ ]:
# rag fájl megnyitása:
with open('/content/drive/MyDrive/Colab Notebooks/cbt_knowledge_base.py', 'r', encoding='utf-8') as f:
    print(f.read())

CBT_KNOWLEDGE_BASE = [
    {
        "id": "cbt_01",
        "category": "circadian",
        "title": "Lefekvési és ébredési idő",
        "text": (
            "Mindig azonos időben keljen fel és feküdjön le, hétvégén is. "
            "A nagyon szabálytalan alvás-ébrenléti időbeosztás gyengíti a belső óra jeleit, és olyan, "
            "mintha gyakran időeltolódást (jet lag) élne át. Különösen fontos az állandó ébredési és "
            "felkelési idő, mert a belső óra a fényjelekből állítja be magát naponta: ha minden nap "
            "ugyanakkor éri fény, az erős és szabályos ritmust ad az órának, ami támogatja a jó alvást."
        ),
        "origin": "terapeuta + Manber et al. (2014)",
        "source": "Alvásterapeuta anyaga; Manber et al. (2014), 1. fejezet A3, p. 19",
    },
    {
        "id": "cbt_02",
        "category": "sleep_hygiene",
        "title": "Nappali alvás hatása",
        "text": (
            "Ne aludjon napközben, és ne pótolja az éjszaka kiesett pihenés

In [ ]:
import sys
import importlib
import pandas as pd

drive_folder = '/content/drive/MyDrive/Colab Notebooks'
if drive_folder not in sys.path:
    sys.path.append(drive_folder)

#Ha a Python korábban gyorsítótárazta a régi/üres verziót, töröljük a memóriából
if 'cbt_knowledge_base' in sys.modules:
   del sys.modules['cbt_knowledge_base']

from cbt_knowledge_base import CBT_KNOWLEDGE_BASE

print(f"Sikeres importálás! Betöltött elemek száma: {len(CBT_KNOWLEDGE_BASE)}")

Sikeres importálás! Betöltött elemek száma: 18


In [ ]:
!pip install -q rank-bm25 sentence-transformers scikit-learn

In [ ]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Korpusz szövegek és azonosítók előkészítése
corpus_texts = [doc["text"] for doc in CBT_KNOWLEDGE_BASE]
corpus_ids = [doc["id"] for doc in CBT_KNOWLEDGE_BASE]

# 2. BM25 (Lexikális kereső) felkészítése
tokenized_corpus = [doc.lower().split() for doc in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)

# 3. Dense beágyazó modell betöltése és indexelés
print("Beágyazó modell betöltése (intfloat/multilingual-e5-base)...")
embedder = SentenceTransformer("intfloat/multilingual-e5-base")

# Az E5 modell elvárása: dokumentumok elé 'passage: ' prefix kell
doc_embeddings = embedder.encode([f"passage: {t}" for t in corpus_texts], show_progress_bar=True)

# 4. Hibrid kereső függvény
def hybrid_search(query_str, alpha=0.5, top_k=2):
    """
    Keresés lexikális (BM25) és sűrű szemantikus (multilingual-e5) pontszámok kombinálásával.
    alpha: 0.0 -> csak BM25 (baseline)
           1.0 -> csak Dense (E5)
           0.5 -> hibrid kombináció
    """
    # BM25 pontszámítás és normalizálás [0, 1] tartományra
    bm25_raw = np.array(bm25.get_scores(query_str.lower().split()))
    if bm25_raw.max() > 0:
        bm25_norm = bm25_raw / bm25_raw.max()
    else:
        bm25_norm = bm25_raw

    # Dense koszinusz-hasonlóság számítás ('query: ' prefixszel)
    q_emb = embedder.encode([f"query: {query_str}"])
    dense_raw = cosine_similarity(q_emb, doc_embeddings)[0]
    dense_norm = (dense_raw + 1) / 2  # skálázás [0, 1] közé

    # Súlyozott hibrid pontszám
    final_scores = (1 - alpha) * bm25_norm + alpha * dense_norm
    top_indices = np.argsort(final_scores)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "doc": CBT_KNOWLEDGE_BASE[idx],
            "score": float(final_scores[idx]),
            "bm25_score": float(bm25_norm[idx]),
            "dense_score": float(dense_norm[idx])
        })
    return results

print("Hibrid indexelés sikeresen befejeződött!")

Beágyazó modell betöltése (intfloat/multilingual-e5-base)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Hibrid indexelés sikeresen befejeződött!


In [ ]:
test_query = "A munka miatt sokáig dolgozok, sokat kávézok és nem tudok elaludni"
talalatok = hybrid_search(test_query, alpha=0.5, top_k=2)

print(f"\nKeresési kifejezés: '{test_query}'\n")
for i, t in enumerate(talalatok, 1):
    doc = t["doc"]
    print(f"{i}. Találat: [{doc['id']}] {doc['title']} (Hibrid pont: {t['score']:.3f})")
    print(f"   Részlet: {doc['text'][:120]}...\n")


Keresési kifejezés: 'A munka miatt sokáig dolgozok, sokat kávézok és nem tudok elaludni'

1. Találat: [cbt_12] Éjszakai ébrenlét (Hibrid pont: 0.962)
   Részlet: Ha azt érzi, már jó ideje nem tud elaludni (vagy visszaaludni), keljen fel! De ne egyen, ne igyon. Két dolgot tehet ilye...

2. Találat: [cbt_11] Aggódás és teendőlista (Hibrid pont: 0.926)
   Részlet: Ha este sokat aggódik, vezessen be egy napközbeni „aggódás félórát”. Segíthet az is, ha a nap végén leírja, mit kell meg...



In [ ]:
#!pip install -q transformers torch accelerate

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-3B-Instruct"


print("Qwen modell és tokenizer betöltése...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
print("Modell sikeresen betöltve a GPU-ra!")

Qwen modell és tokenizer betöltése...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Modell sikeresen betöltve a GPU-ra!


In [ ]:

def generate_cbt_advice(query_str, retrieved_results):

    context_chunks = []
    target_ids = []

    for r in retrieved_results:
        doc = r.get("doc", r)

        c_id = doc.get("id", "cbt")
        c_title = doc.get("title", "")
        c_text = doc.get("text", "")

        if not c_text.strip():
            continue

        context_chunks.append(
            f"[{c_id}] {c_title}\n{c_text}"
        )
        target_ids.append(c_id)

    if not context_chunks:
        return "Nem találtam releváns forrást."

    context_text = "\n\n".join(context_chunks)

    system_prompt = """
Te egy magyar nyelvű alvástanácsadó asszisztens vagy.
A megadott CBT-I források alapján válaszolj.
Ne találj ki információkat.
Írj természetes, egyszerű magyar mondatokat.
Ne ismételd ugyanazt a gondolatot.
Minden forráshoz egy külön, konkrét tanácsot adj.
A forrás azonosítóját tüntesd fel KÖTELEZŐEN minden pont végére szögletes zárójelben (pl. [cbt_11])!\n"
"""

    user_prompt = f"""
KÉRDÉS:
{query_str}

FORRÁSOK:
{context_text}

Készíts pontosan {len(context_chunks)} számozott pontot.
Minden pont 1-2 mondat legyen.
Mindegyik pont más tanácsot tartalmazzon.
A választ teljes mondatokkal fejezd be. A forrás azonosítóját tüntesd fel KÖTELEZŐEN minden pont végére szögletes zárójelben.
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        [formatted_prompt],
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_tokens = generated_ids[0][
        inputs.input_ids.shape[1]:
    ]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()


In [ ]:
#Tesztkérdés
test_issue = "Állandóan forgolódom az ágyban, nem tudok elaludni és folyamatosan nézem az órát, hogy hány órám van még hátra reggelig."

#Top 2 találat a tudásbázisból
retrieved = hybrid_search(test_issue, alpha=0.5, top_k=2)

print("--- VISSZAKERESETT FORRÁSOK ---")
for r in retrieved:
    print(f"[{r['doc']['id']}] {r['doc']['title']} (Pont: {r['score']:.3f})")

#Generálás a Qwen modellel
print("\n--- GENERÁLT TANÁCS (QWEN RAG) ---")
advice = generate_cbt_advice(test_issue, retrieved)
print(advice)

--- VISSZAKERESETT FORRÁSOK ---
[cbt_13] Forgolódás és óranézés (Pont: 0.971)
[cbt_12] Éjszakai ébrenlét (Pont: 0.775)

--- GENERÁLT TANÁCS (QWEN RAG) ---
[cbt_13]
Forgolódás helyett próbálja meg, hogy elaludjon. Ha nem tudja elaludni, akkor mindig visszafekszik, és kérdezze, mennyire lassan elhaladna a reggelig. Ez segíthet az órát nézni és a fejszámolást végezni.

[cbt_12]
Ha elhaladna a reggelig, akkor mindig visszafekszik, és olvaszon egy kicsit, majd visszafekszik. Ne próbáljon aludni, csak akkor menjen vissza az ágyba, ha álmos. Kerülje az erős fényt, és ne a kanapén aludjon át az éjszakát. A legfontosabb, hogy ne próbáljon aludni, hanem elkezdis elaludni, mivel ez a fáradtság alacsony energia gyakorolja.


In [ ]:
test_queries = [
    {"q": "Miért fontos hétvégén is pontosan ugyanabban az időpontban felkelni reggel?", "expected_id": "cbt_01"},
    {"q": "Szabad-e délután aludni vagy szunyókálni egyet, ha fáradtnak érzem magam?", "expected_id": "cbt_02"},
    {"q": "Szabad-e a hálószobában, az ágyban fekve dolgozni, tévézni vagy telefonozni?", "expected_id": "cbt_03"},
    {"q": "Hány óráig szabad kávét vagy koffeintartalmú italokat inni napközben?", "expected_id": "cbt_04"},
    {"q": "Hogyan hat a lefekvés előtti alkoholfogyasztás és a nehéz vacsora az éjszakai alvásra?", "expected_id": "cbt_05"},
    {"q": "Mikor szabad sportolni és intenzív edzést végezni este elalvás előtt?", "expected_id": "cbt_06"},
    {"q": "Milyen a megfelelő hálószobai környezet a jó alváshoz?", "expected_id": "cbt_09"},
    {"q": "Hogyan alakítsak ki esti lefekvési rituálét és puffer időt a feszültség levezetésére?", "expected_id": "cbt_10"},
    {"q": "Mit tegyek konkrétan, ha éjszaka felébredek és huzamosabb ideig nem tudok visszaaludni?", "expected_id": "cbt_12"},
    {"q": "Miért tilos az órát nézegetni és perceket számolni, amikor az ágyban forgolódom?", "expected_id": "cbt_13"}
]

In [ ]:
def evaluate_retriever(alpha_val, k=2):
    hits = 0
    reciprocal_ranks = []

    for item in test_queries:
        results = hybrid_search(item["q"], alpha=alpha_val, top_k=k)
        retrieved_ids = [r["doc"]["id"] for r in results]

        #Recall@
        if item["expected_id"] in retrieved_ids:
            hits += 1
            # Rang megkeresése az MRR-hez
            rank = retrieved_ids.index(item["expected_id"]) + 1
            reciprocal_ranks.append(1.0 / rank)
        else:
            reciprocal_ranks.append(0.0)

    recall_at_k = hits / len(test_queries)
    mrr = np.mean(reciprocal_ranks)
    return recall_at_k, mrr

#Mérések
bm25_rec, bm25_mrr = evaluate_retriever(alpha_val=0.0, k=2)
dense_rec, dense_mrr = evaluate_retriever(alpha_val=1.0, k=2)
hybrid_rec, hybrid_mrr = evaluate_retriever(alpha_val=0.5, k=2)

print("=== VISSZAKERESÉSI ÉRTÉKELÉS (TOP-2) ===")
print(f"1. BM25 Baseline   : Recall@2 = {bm25_rec:.2f} | MRR = {bm25_mrr:.2f}")
print(f"2. Dense E5        : Recall@2 = {dense_rec:.2f} | MRR = {dense_mrr:.2f}")
print(f"3. Hibrid (BM25+E5): Recall@2 = {hybrid_rec:.2f} | MRR = {hybrid_mrr:.2f}")

=== VISSZAKERESÉSI ÉRTÉKELÉS (TOP-2) ===
1. BM25 Baseline   : Recall@2 = 0.70 | MRR = 0.65
2. Dense E5        : Recall@2 = 1.00 | MRR = 1.00
3. Hibrid (BM25+E5): Recall@2 = 0.70 | MRR = 0.65


In [ ]:
def generate_no_rag_baseline(user_query):
    """Közvetlen promptolás CBT-I kontextus nélkül (Baseline)."""
    messages = [
        {"role": "system", "content": "Te egy alvási tanácsadó vagy. Válaszolj a kérdésre röviden és tömören!"},
        {"role": "user", "content": user_query}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=200, temperature=0.2)
    new_tokens = out_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

#Összehasonlító futtatás egy tesztkérdésre:
sample_q = "Mit csináljak pontosan, ha hajnali 3-kor felébredek és nem jön álom a szememre?"

#Alapvonal (Baseline)
no_rag_ans = generate_no_rag_baseline(sample_q)

#RAG rendszer
rag_results = hybrid_search(sample_q, alpha=0.5, top_k=2)
rag_ans = generate_cbt_advice(sample_q, rag_results)

print("--- 1. ALAPVONAL (NO-RAG BASELINE) ---")
print(no_rag_ans)
print("\n--- 2. RAG MEGOLDÁS (FORRÁSHIVATKOZÁSSAL) ---")
print(rag_ans)

--- 1. ALAPVONAL (NO-RAG BASELINE) ---
Nagyon fontos, hogy gyorsan elhagyaljonak a hajnal, hogy az álom eltüntethessék. Mindig érdemes kicsit időt veszteni a szemeid mellett, hogy megfigyelhessük a fogyasztást.

--- 2. RAG MEGOLDÁS (FORRÁSHIVATKOZÁSSAL) ---
Pont 1:

[KCBT_17] Ha hajnal 3-4 óra előtt felébredek és nem jön álom a szememre, azt javaslom, hogy néha egy napközben „belső óra félórát” vezess be. Ez segíthet az ébredéshez és az alvásnyomáshoz, mert ez a napközbeni tárgyalás segíthet az ébredéshez és az alvásnyomáshoz. 

Pont 2:

[KCBT_17] Azonnal néha 1-2 órájuk múlva, és még mindig nincs álom, akkor próbáljunk meg, hogy az ébredéshez és az alvásnyomáshoz használjunk egy napközbeni tárgyalást. Ez lehetne egy olyan eszköz, ami segíthet az ébredéshez és az alvásnyomáshoz. 

[KCBT_11] Ha aggódhatok, néha vezessen be egy napközbeni „aggódás félórát”, amely segíthet az ébredéshez és az alvásnyomáshoz. Az írásos terv


In [ ]:
sleep_data=pd.read_csv("/content/drive/My Drive/Egyetem/Diplomamunka/Source/sleep_health_dataset.csv")
sleep_data.head()

,person_id,age,gender,occupation,bmi,country,sleep_duration_hrs,sleep_quality_score,rem_percentage,deep_sleep_percentage,...,heart_rate_resting_bpm,sleep_aid_used,shift_work,room_temperature_celsius,weekend_sleep_diff_hrs,season,day_type,cognitive_performance_score,sleep_disorder_risk,felt_rested
0,1,29,Female,Driver,25.7,Japan,6.19,6.6,22.5,19.3,...,63,0,0,20.1,1.84,Autumn,Weekday,73.4,Healthy,0
1,2,55,Female,Software Engineer,22.0,USA,8.32,6.9,26.9,14.9,...,52,1,0,18.0,0.13,Winter,Weekend,99.4,Healthy,1
2,3,42,Male,Nurse,25.0,India,3.74,1.0,20.2,16.2,...,72,0,1,17.9,1.67,Spring,Weekend,2.5,Severe,0
3,4,37,Female,Student,29.5,India,6.79,6.4,17.7,17.7,...,71,0,0,19.1,2.37,Summer,Weekend,67.8,Healthy,0
4,5,23,Male,Lawyer,23.6,Spain,5.02,3.2,23.3,18.3,...,71,0,0,19.7,1.26,Summer,Weekday,38.1,Mild,0


In [ ]:
#Nem numerikus adatok átalakítása
sleep_data = pd.get_dummies(sleep_data, columns=['gender'], drop_first=True)
sleep_data['is_weekend'] = sleep_data['day_type'].map({'Weekday': 0, 'Weekend': 1})
sleep_data['chronotype_code'] = sleep_data['chronotype'].map({'Morning': 0, 'Neutral': 1, 'Evening': 2})
risk_mapping = {
    'Healthy': 0,
    'Mild': 1,
    'Moderate': 2,
    'Severe': 3
}

sleep_data['sleep_disorder_risk_score'] = sleep_data['sleep_disorder_risk'].map(risk_mapping)

def classify_activity(row):
    steps = row.get('steps_that_day', 0)
    exercise = row.get('exercise_day', 0)

    # 2: Magas aktivitás (edzésnap VAGY legalább 10 000 lépés)
    if exercise == 1 or steps >= 10000:
        return 2
    # 1: Mérsékelt aktivitás (nincs edzés, de 5 000 - 9 999 lépés megvan)
    elif steps >= 5000:
        return 1
    # 0: Mozgásszegény (nincs edzés és < 5 000 lépés)
    else:
        return 0

# Új oszlop létrehozása a sleep_data DataFrame-ben:
sleep_data['activity_level'] = sleep_data.apply(classify_activity, axis=1)

print(sleep_data['activity_level'].value_counts().sort_index())

activity_level
0    21512
1    28952
2    49536
Name: count, dtype: int64


In [ ]:
def generate_user_profile_and_query(row):
    """
    Egyetlen személy keresztmetszeti rekordjából összeállítja
    a demográfiai hátteret és a fennálló alvásproblémákat,
    majd RAG lekérdezéssé formálja.
    """
    # 1. Személyes / Demográfiai háttér
    profile_info = []

    if 'age' in row and pd.notnull(row['age']):
        profile_info.append(f"{int(row['age'])} éves")

    if 'gender' in row and pd.notnull(row['gender']):
        profile_info.append(str(row['gender']).lower())
    elif row.get('gender_Male', 0) == 1:
        profile_info.append("férfi")
    elif row.get('gender_Other', 0) == 1:
        profile_info.append("egyéb nemű")

    if 'bmi' in row and pd.notnull(row['bmi']):
        profile_info.append(f"BMI: {row['bmi']:.1f}")

    if 'chronotype' in row and pd.notnull(row['chronotype']):
        profile_info.append(f"{row['chronotype']} kronotípus")
    elif 'chronotype_code' in row:
        chrono_map = {0: "korán kelő (pacsirta)", 1: "köztes", 2: "esti (bagoly)"}
        profile_info.append(f"{chrono_map.get(row['chronotype_code'], 'átlagos')} kronotípus")

    profile_desc = ", ".join(profile_info) if profile_info else "nem részletezett profil"

    # 2. Megfigyelhető alvási nehézségek és életmódbeli kockázatok
    observed_issues = []

    # Kockázati kategória
    risk_score = row.get('sleep_disorder_risk_score', 0)
    if risk_score >= 2:
        observed_issues.append("orvosilag mérhető alvászavar-kockázat")
    elif risk_score == 1:
        observed_issues.append("enyhe alvásminőség-romlási hajlam")

    # Alvásmennyiség
    sleep_hrs = row.get('sleep_duration_hrs', 8.0)
    if sleep_hrs < 6.0:
        observed_issues.append(f"krónikusan kevés alvásidő ({sleep_hrs:.1f} óra)")
    elif sleep_hrs < 6.5:
        observed_issues.append(f"átlag alatti alvásmennyiség ({sleep_hrs:.1f} óra)")

    # Stressz-szint
    stress = row.get('stress_score', 0)
    if stress >= 7:
        observed_issues.append(f"kiemelkedően magas stressz-szint ({stress}/10)")

    # Elalvási idő (latencia)
    latency = row.get('sleep_latency_mins', 15)
    if latency > 30:
        observed_issues.append(f"elnyújtott elalvási idő és ágyban forgolódás ({latency} perc)")

    # Éjszakai ébredések
    wakes = row.get('wake_episodes_per_night', 0)
    if wakes >= 2:
        observed_issues.append(f"töredezett alvás ({wakes} éjszakai ébredés)")

    # Fizikai aktivitási szint
    act = row.get('activity_level', 1)
    if act == 0:
        observed_issues.append("jellemzően mozgásszegény életmód")

    # Életmódbeli stimulánsok / szokások
    if row.get('screen_time_before_bed_mins', 0) > 60:
        observed_issues.append(f"jelentős esti képernyőhasználat ({row['screen_time_before_bed_mins']} perc)")
    if row.get('caffeine_mg_before_bed', 0) > 50:
        observed_issues.append("késő esti koffeinbevitel")
    if row.get('alcohol_units_before_bed', 0) >= 2:
        observed_issues.append("lefekvés előtti alkoholfogyasztás")

    # Munkarend
    if row.get('shift_work', 0) == 1:
        observed_issues.append("váltott műszakos munkarend miatti rendszertelen alvásciklus")
    elif row.get('work_hours_that_day', 8) > 9:
        observed_issues.append(f"hosszú napi munkaidő ({row['work_hours_that_day']} óra)")

    # 3. Kérdés összeállítása
    if observed_issues:
        issues_text = "; ".join(observed_issues)
        query = (
            f"Adott egy {profile_desc} személy, akinél a következő alvási és életmódbeli tényezők figyelhetők meg: {issues_text}. "
            "Milyen konkrét kognitív viselkedésterápiás (CBT-I) protokollok és alváshigiénés szabályok javasoltak számára az alvásminőség javítására?"
        )
    else:
        query = (
            f"Adott egy {profile_desc} személy, akinél nem állnak fenn jelentős panaszok. "
            "Milyen megelőző alváshigiénés szabályok segítenek az optimális alváshatékonyság fenntartásában?"
        )

    return {
        "profile": profile_desc,
        "issues": observed_issues,
        "rag_query": query
    }

In [ ]:
# 1. Kiválasztunk egy tetszőleges sort a táblázatból (pl. a legelő pácienst)
patient_row = sleep_data.iloc[4]

# 2. Generáljuk a személyes profilt és a strukturált RAG lekérdezést
patient_data = generate_user_profile_and_query(patient_row)
auto_query = patient_data["rag_query"]

print("=" * 70)
print(f"PROFIL: {patient_data['profile']}")
print("AZONOSÍTOTT TÜNETEK / FAKTOROK:")
for issue in patient_data["issues"]:
    print(f"  • {issue}")
print(f"\nAUTOMATIKUSAN GENERÁLT KÉRDÉS:\n{auto_query}")
print("=" * 70)

# -------------------------------------------------------------
# 3. ÖSSZEHASONLÍTÁS: BASELINE (NO-RAG) VS. RAG RENDSZER
# -------------------------------------------------------------

# A) Alapvonal (Baseline) közvetlen promptolással:
print("\n--- 1. ALAPVONAL (NO-RAG BASELINE) ---")
no_rag_result = generate_no_rag_baseline(auto_query)
print(no_rag_result)

# B) Hibrid keresés a CBT-I korpuszon:
print("\n--- 2. HIBRID KERESŐ TALÁLATAI (TOP-2 CHUNK) ---")
retrieved_results = hybrid_search(auto_query, alpha=0.5, top_k=2)

for i, result in enumerate(retrieved_results):
    doc = result["doc"]
    score = result["score"]
    print(f"  [{doc['id']}] {doc['title']} (Fúziós pont: {score:.4f})")

# C) Válaszgenerálás a Qwennek:
print("\n--- 3. RAG MEGOLDÁS (FORRÁSHIVATKOZÁSSAL) ---")
rag_result = generate_cbt_advice(auto_query, retrieved_results)
print(rag_result)

PROFIL: 23 éves, férfi, BMI: 23.6, Neutral kronotípus
AZONOSÍTOTT TÜNETEK / FAKTOROK:
  • enyhe alvásminőség-romlási hajlam
  • krónikusan kevés alvásidő (5.0 óra)
  • kiemelkedően magas stressz-szint (7.4/10)
  • töredezett alvás (5 éjszakai ébredés)
  • jellemzően mozgásszegény életmód
  • jelentős esti képernyőhasználat (72 perc)
  • hosszú napi munkaidő (10.4 óra)

AUTOMATIKUSAN GENERÁLT KÉRDÉS:
Adott egy 23 éves, férfi, BMI: 23.6, Neutral kronotípus személy, akinél a következő alvási és életmódbeli tényezők figyelhetők meg: enyhe alvásminőség-romlási hajlam; krónikusan kevés alvásidő (5.0 óra); kiemelkedően magas stressz-szint (7.4/10); töredezett alvás (5 éjszakai ébredés); jellemzően mozgásszegény életmód; jelentős esti képernyőhasználat (72 perc); hosszú napi munkaidő (10.4 óra). Milyen konkrét kognitív viselkedésterápiás (CBT-I) protokollok és alváshigiénés szabályok javasoltak számára az alvásminőség javítására?

--- 1. ALAPVONAL (NO-RAG BASELINE) ---
A konkrét CBT-I protokol